# 02 — Verified Deterministic Forecast Panel

This notebook constructs the deterministic forecast and HKO outcome
panel used by the probabilistic post-processing models.

Only forecast paths that can be independently certified from 24
Hong Kong local hourly forecasts are retained.

In [1]:
from pathlib import Path
import json

import pandas as pd

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "data/manifests/"
            "02_verified_full_panel_manifest.json"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

selected = pd.read_csv(
    ROOT
    / "data/processed/"
    "02_selected_deterministic_forecast_panel.csv"
)

training = pd.read_csv(
    ROOT
    / "data/processed/"
    "02_weather_training_panel.csv"
)

reconciliation = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "02_historical_daily_hourly_reconciliation.csv"
)

support = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "02_date_rule_support_matrix.csv"
)

summary = json.loads(
    (
        ROOT
        / "outputs/diagnostics/"
        "02_full_support_summary.json"
    ).read_text(encoding="utf-8")
)

print("Status:", summary["status"])
print("Verified rows:", len(selected))
print("Verified dates:", selected["target_date"].nunique())
print("Historical rows:", summary["historical_verified_rows"])
print("June rows:", summary["june_verified_rows"])

Status: NOTEBOOK02_VERIFIED_PANEL_READY
Verified rows: 375
Verified dates: 102
Historical rows: 256
June rows: 119


## Historical forecast identification

The stored column `forecast_hko_daily_max_C` is a forecast of the
daily maximum at the HKO location. It is not the realised HKO target.

On all 256 independently reconstructed historical paths, the stored
value and the maximum of the 24 hourly forecasts agree exactly.

In [2]:
assert len(reconciliation) == 256

assert reconciliation[
    "unique_local_hours"
].eq(24).all()

assert reconciliation[
    "absolute_difference_c"
].le(1e-9).all()

print(
    "Maximum stored-versus-reconstructed discrepancy:",
    reconciliation[
        "absolute_difference_c"
    ].max(),
)

Maximum stored-versus-reconstructed discrepancy: 0.0


## Information-time condition

Every forecast issue time must be no later than the applicable
decision cutoff. Settlement date is used as the uncertainty unit.

In [3]:
issue = pd.to_datetime(
    selected["forecast_issue_time_utc"],
    utc=True,
)

decision = pd.to_datetime(
    selected["decision_time_utc"],
    utc=True,
)

assert issue.notna().all()
assert decision.notna().all()
assert (issue <= decision).all()

assert not selected[
    ["target_date", "decision_rule"]
].duplicated().any()

print(
    "All forecasts available by decision time:",
    bool((issue <= decision).all()),
)

All forecasts available by decision time: True


## Evidential boundary

Thirty-six March-May request rows are not promoted into the empirical
sample because no complete independently verified hourly path is
available. One June date-rule combination is also unavailable.

The retained panel therefore contains 375 date-rule observations over
102 settlement dates.

In [4]:
missing = support.loc[
    ~support["forecast_present"],
    [
        "target_date",
        "decision_rule",
        "support_status",
    ],
]

print(
    missing[
        "support_status"
    ].value_counts().to_string()
)

assert len(missing) == 37
assert len(selected) == 375
assert selected["target_date"].nunique() == 102

support_status
NO_INDEPENDENTLY_VERIFIED_HISTORICAL_PATH    36
NO_ADMISSIBLE_JUNE_PATH                       1


In [5]:
support_by_period = (
    selected.groupby("source_period")
    .agg(
        rows=("target_date", "size"),
        dates=("target_date", "nunique"),
        start_date=("target_date", "min"),
        end_date=("target_date", "max"),
    )
    .reset_index()
)

print(support_by_period.to_string(index=False))

       source_period  rows  dates start_date   end_date
historical_march_may   256     72 2026-03-16 2026-05-31
       june_external   119     30 2026-06-01 2026-06-30
